# Overview

This script is designed to perform a series of simulations using SWMM (Storm Water Management Model) with different rain shift scenarios.  
It processes storm data, updates time series inputs for each simulation, runs the SWMM model, and extracts key results from the output.  
The script supports multiple event types, such as 'historical' and 'future', and calculates various metrics including outflows, rainfall intensities, and runoff coefficients for different storm events.  
It also includes functionalities for dynamically adjusting the input files based on specific spatial shifts (`x_shift`, `y_shift`) to simulate different rain patterns and observe their impact on runoff.  
Results are collected, processed, and stored in a DataFrame with a MultiIndex structure for easy analysis. The final results are saved in a pickle file for further use.

## Key Components:
- **Data Loading**: Loads the SWMM output file and parses it into a DataFrame for analysis.
- **Rain Shift Processing**: Reads rain shift text files and updates the time series data in the SWMM input file accordingly.
- **Statistical Analysis**: Computes key metrics like total outflow, rainfall intensities, and runoff coefficients based on the storm data.
- **Result Storage**: Results are stored in a structured DataFrame with hierarchical column names and saved in a pickle file.

The script is organized to handle multiple combinations of spatial shifts, event types, and storm scenarios, making it adaptable to different simulation needs.


## Imports and files

In [1]:
import os
import pickle
import re
import itertools

# Date and time related imports
from datetime import datetime, timedelta

# Scientific libraries
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.interpolate import griddata

# Geospatial libraries
import geopandas as gpd
import contextily as ctx
from shapely.geometry import Polygon, Point

# Plotting libraries
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.lines import Line2D

# SWMM related imports
from swmm_api.input_file import read_inp_file, SwmmInput, section_labels as sections
from swmm_api import read_out_file, read_rpt_file, swmm5_run, SwmmReport

## Functions

In [2]:
def read_rain_shift_txt_file(rain_shift_directory_path, x_shift, y_shift):
    # List all files in the directory
    files_in_directory = os.listdir(rain_shift_directory_path)

    # Filter files with the desired format
    matching_files = [file for file in files_in_directory if "rain_shift_" in file]

    if not matching_files:
        print("No matching files found.")
        return None, None, None

    # Find the file that matches the specified x_shift and y_shift
    x_shift_str = str(abs(x_shift))
    y_shift_str = str(abs(y_shift))
    matching_file = None
    for file in matching_files:
        x_match = re.search(r'x(plus|minus)_(\d+)', file)
        y_match = re.search(r'y(plus|minus)_(\d+)', file)
        
        if x_match and y_match:
            file_x_shift = int(x_match.group(2)) * (1 if x_match.group(1) == 'plus' else -1)
            file_y_shift = int(y_match.group(2)) * (1 if y_match.group(1) == 'plus' else -1)
            if file_x_shift == x_shift and file_y_shift == y_shift:
                matching_file = file
                break

    if not matching_file:
        print(f"No file found with x_shift: {x_shift} and y_shift: {y_shift}")
        return None, None, None

    # Construct the file path
    file_path = os.path.join(rain_shift_directory_path, matching_file)

    # Open and read the contents of the matched file
    with open(file_path, 'r') as file:
        rain_shift_txt = file.read()

    # Return x_shift, y_shift, and file_contents
    return rain_shift_txt

In [3]:
def parse_date_time(row):
    """
    Parse date and time from a formatted row.
    """
    columns = row.split()
    date_str = columns[1]
    time_str = columns[2]
    date = datetime.strptime(date_str, '%m/%d/%Y').date()
    time = datetime.strptime(time_str, '%H:%M:%S').time()
    return date, time

def update_TimeSeriesData(inp, timeseries_text):
    """
    Update the time series data in an input file used for simulation.
    
    Args:
    - inp (dict): Dictionary containing simulation input data.
    - timeseries_text (str): Text containing transformed time series data.
    
    Returns:
    - None
    """
    lines = timeseries_text.strip().split('\n')
    data_rows = [line for line in lines if not line.startswith(';;')]
    
    # Parse the first and last data rows for start and end times
    first_date, first_time = parse_date_time(data_rows[0])
    second_date, second_time = parse_date_time(data_rows[1])
    last_date, last_time = parse_date_time(data_rows[-1])
    
    first_datetime = datetime.combine(first_date, first_time)
    second_datetime = datetime.combine(second_date, second_time)
    
    # Calculate time interval between the first and second rows
    first_timedelta = timedelta(hours=first_time.hour, minutes=first_time.minute, seconds=first_time.second)
    second_timedelta = timedelta(hours=second_time.hour, minutes=second_time.minute, seconds=second_time.second)
    
    # Calculate the difference
    time_interval = str(second_timedelta - first_timedelta)
    
    # Calculate start and end times
    start_datetime = datetime.combine(first_date, first_time)
    if start_datetime.hour < 6:
        start_datetime -= timedelta(days=1)
    START_DATE = start_datetime.date()
    REPORT_START_DATE = START_DATE
    START_TIME = (start_datetime - timedelta(hours=6)).time()
    REPORT_START_TIME = START_TIME
    
    end_datetime = datetime.combine(last_date, last_time)
    if end_datetime.hour >= 18:
        end_datetime += timedelta(days=1)
    END_DATE = end_datetime.date()
    END_TIME = (end_datetime + timedelta(hours=6)).time()
    
    # Update the options section with calculated times
    inp['OPTIONS']['START_DATE'] = START_DATE
    inp['OPTIONS']['START_TIME'] = START_TIME
    inp['OPTIONS']['REPORT_START_DATE'] = REPORT_START_DATE
    inp['OPTIONS']['REPORT_START_TIME'] = REPORT_START_TIME
    inp['OPTIONS']['END_DATE'] = END_DATE
    inp['OPTIONS']['END_TIME'] = END_TIME
    
    # Update the time series section with new data
    inp['TIMESERIES'] = timeseries_text

    # Ensure that the number of time series matches the number of rain gauges
    if len(inp['TIMESERIES'].keys()) != len(inp['RAINGAGES'].keys()):
        raise ValueError('ERROR: The number of time series does not match the number of rain gauges.')

    # Edit the time series name and its corresponding rain gauge field
    for basin_timeserie in range(len(list(inp['TIMESERIES'].keys()))):
        basin_timeserie_name = inp['TIMESERIES'][list(inp['TIMESERIES'].keys())[basin_timeserie]]['name']
        inp['RAINGAGES'][list(inp['RAINGAGES'].keys())[basin_timeserie]]['timeseries'] = basin_timeserie_name
        inp['RAINGAGES'][list(inp['RAINGAGES'].keys())[basin_timeserie]]['interval'] = time_interval
        inp['RAINGAGES'][list(inp['RAINGAGES'].keys())[basin_timeserie]]['form'] = "INTENSITY"


In [4]:
def update_imperviousness_with_factor(subcatchment_dict, factor):
    """
    Update the imperviousness values of SubCatchment objects in a dictionary by a factor.

    Args:
        subcatchment_dict (dict): A dictionary containing SubCatchment objects as values with subcatchment names as keys.
        factor (float): The factor by which imperviousness values need to be powered.

    Returns:
        dict: The updated dictionary with imperviousness values powered by the factor.
    """


    # Loop through each subcatchment in the dictionary
    for index, (subcatchment_name, subcatchment) in enumerate(subcatchment_dict.items()):
#         initial_imperviousness = impervious_initial_values[index]
#         subcatchment.imperviousness = initial_imperviousness
        powered_imperviousness = subcatchment.imperviousness * factor
        if powered_imperviousness > 100:
            powered_imperviousness =100
        # Update the imperviousness value for the current subcatchment with the powered imperviousness value
        subcatchment.imperviousness = powered_imperviousness


    # Return the updated dictionary
    return subcatchment_dict

In [5]:
def calculate_mean_wet_area_percent(inp, rain_threshold=0.1):
    """
    Calculate the average percentage of the basin area that receives rainfall 
    above a given threshold throughout the simulation.
    """
    subcatchments = inp[sections.SUBCATCHMENTS]
    timeseries_dict = inp['TIMESERIES']

    # Extract the common prefix (e.g., '19911102') from any key
    sample_key = next(iter(timeseries_dict.keys()))
    prefix = sample_key.split('_')[0]  # Will give '19911102'

    mapping = []
    for name, obj in subcatchments.items():
        try:
            # Build gage name from subcatchment name (e.g., S01 → 19911102_S1)
            subcatch_num = name.lstrip('S0') or name.lstrip('S')
            rain_gage = f"{prefix}_S{subcatch_num}"
            area = float(getattr(obj, 'Area', None) or getattr(obj, 'area', None))

            if rain_gage in timeseries_dict:
                timeseries = timeseries_dict[rain_gage].data
                mapping.append((name, rain_gage, area, timeseries))
        except Exception:
            continue

    if not mapping:
        return np.nan

    time_steps = [entry[0] for entry in mapping[0][3]]
    wet_area_percent = []

    for i in range(len(time_steps)):
        wet_area = 0
        total_area = 0

        for _, _, area, series in mapping:
            rain = series[i][1]
            total_area += area
            if rain > rain_threshold:
                wet_area += area

        wet_area_percent.append(wet_area / total_area * 100)

    return np.mean(wet_area_percent)


def calculate_wet_area_percent_stats(inp, rain_threshold=0.1):
    """
    Calculate both the mean and maximum percentage of the basin area that receives
    rainfall above a given threshold throughout the simulation.

    Returns
    -------
    (mean_percent, max_percent): tuple of floats
        - mean_percent: Mean wet area percent over time
        - max_percent: Maximum wet area percent observed over time
    """
    subcatchments = inp[sections.SUBCATCHMENTS]
    timeseries_dict = inp['TIMESERIES']

    # Extract the common prefix (e.g., '19911102') from any key
    try:
        sample_key = next(iter(timeseries_dict.keys()))
    except StopIteration:
        return np.nan, np.nan
    prefix = sample_key.split('_')[0]

    mapping = []
    for name, obj in subcatchments.items():
        try:
            # Build gage name from subcatchment name (e.g., S01 → 19911102_S1)
            subcatch_num = name.lstrip('S0') or name.lstrip('S')
            rain_gage = f"{prefix}_S{subcatch_num}"
            area = float(getattr(obj, 'Area', None) or getattr(obj, 'area', None))

            if rain_gage in timeseries_dict:
                timeseries = timeseries_dict[rain_gage].data
                mapping.append((name, rain_gage, area, timeseries))
        except Exception:
            continue

    if not mapping:
        return np.nan, np.nan

    time_steps = [entry[0] for entry in mapping[0][3]]
    wet_area_percent = []

    for i in range(len(time_steps)):
        wet_area = 0
        total_area = 0

        for _, _, area, series in mapping:
            rain = series[i][1]
            total_area += area
            if rain > rain_threshold:
                wet_area += area

        if total_area > 0:
            wet_area_percent.append(wet_area / total_area * 100)
        else:
            wet_area_percent.append(np.nan)

    # Compute stats robustly
    arr = np.array(wet_area_percent, dtype=float)
    mean_val = float(np.nanmean(arr)) if arr.size else np.nan
    max_val = float(np.nanmax(arr)) if arr.size else np.nan
    return mean_val, max_val

In [6]:
# Define paths and parameters
sim_dir = "D:/Development/RESEARCH/Raanana/SWMM/from_radar/Climate_Change"
EVENTS_L = [f"{i:02}" for i in range(1, 42)]  # Use full range when needed
wrf_time_type = ['historical', 'future']
INP_FILE = r"Final.inp"
RAIN_THRESHOLD = 0.5  # rainfall threshold for wet-area calculations, 1,3,5,10, 15 , 25

# Store results separately
wet_area_records = []

# Loop over events and event types
for filename_event in EVENTS_L:
    for event_type in wrf_time_type:
        print(f"Processing wet area for {event_type} event {filename_event}")
        rainshift_txtfiles_path = fr'\\vscifs\hydrolab1\hydrolab\home\Raz\WRF\txtfiles\{event_type}\event_{filename_event}'
        inp = read_inp_file(os.path.join(sim_dir, INP_FILE))

        for x_shift in np.arange(0, 1, 500):
            for y_shift in np.arange(-20000, 20500, 500):
                d = np.sqrt(x_shift**2 + y_shift**2)

                try:
                    rain_shift_txt = read_rain_shift_txt_file(rainshift_txtfiles_path, x_shift, y_shift)
                    update_TimeSeriesData(inp, rain_shift_txt)
                    mean_wet, max_wet = calculate_wet_area_percent_stats(inp, rain_threshold=RAIN_THRESHOLD)

                    wet_area_records.append({
                        'event_num': filename_event,
                        'x': x_shift,
                        'y': y_shift,
                        'd': d,
                        'event_type': event_type,
                        'wet_area_percent_mean': mean_wet,
                        'wet_area_percent_max': max_wet,
                    })
                except Exception as e:
                    print(f"Failed at event {filename_event} {event_type}, y={y_shift}: {e}")
                    continue

# Convert to DataFrame
wet_area_df = pd.DataFrame(wet_area_records)

# Split by type
df_h = wet_area_df[wet_area_df['event_type'] == 'historical'].copy()
df_f = wet_area_df[wet_area_df['event_type'] == 'future'].copy()

# Drop 'event_type' column
df_h = df_h.drop(columns='event_type')
df_f = df_f.drop(columns='event_type')

# Rename wet_area columns with MultiIndex style
df_h = df_h.rename(columns={
    'wet_area_percent_mean': ('historical', 'spatial_rain', 'wet_area_percent_mean'),
    'wet_area_percent_max': ('historical', 'spatial_rain', 'wet_area_percent_max'),
})
df_f = df_f.rename(columns={
    'wet_area_percent_mean': ('future', 'spatial_rain', 'wet_area_percent_mean'),
    'wet_area_percent_max': ('future', 'spatial_rain', 'wet_area_percent_max'),
})

# Set MultiIndex for all columns
df_h.columns = pd.MultiIndex.from_tuples(
    [col if isinstance(col, tuple) else (col, '', '') for col in df_h.columns]
)
df_f.columns = pd.MultiIndex.from_tuples(
    [col if isinstance(col, tuple) else (col, '', '') for col in df_f.columns]
)

# Merge
wet_area_merged = pd.merge(
    df_h,
    df_f,
    on=[('event_num', '', ''), ('x', '', ''), ('y', '', ''), ('d', '', '')],
    how='outer'
)

# Ensure columns remain MultiIndex
wet_area_merged.columns = pd.MultiIndex.from_tuples(wet_area_merged.columns)

# Save as pickle
pickle_name = f"wet_area_summary_{int(RAIN_THRESHOLD) if isinstance(RAIN_THRESHOLD, (int, np.integer)) or (isinstance(RAIN_THRESHOLD, float) and RAIN_THRESHOLD.is_integer()) else RAIN_THRESHOLD}.pkl"
output_path = os.path.join(sim_dir, 'pickles', pickle_name)
wet_area_merged.to_pickle(output_path)

Processing wet area for historical event 01
Processing wet area for future event 01
Processing wet area for historical event 02
Processing wet area for future event 02
Processing wet area for historical event 03
Processing wet area for future event 03
Processing wet area for historical event 04
Processing wet area for future event 04
Processing wet area for historical event 05
Processing wet area for future event 05
Processing wet area for historical event 06
Processing wet area for future event 06
Processing wet area for historical event 07
Processing wet area for future event 07
Processing wet area for historical event 08
Processing wet area for future event 08
Processing wet area for historical event 09
Processing wet area for future event 09
Processing wet area for historical event 10
Processing wet area for future event 10
Processing wet area for historical event 11
Processing wet area for future event 11
Processing wet area for historical event 12
Processing wet area for future e

In [7]:
sim_dir

'D:/Development/RESEARCH/Raanana/SWMM/from_radar/Climate_Change'